# E2 · Few-shot fine-tuning on real photos (inscription-grouped cross-validation)

**Question.** The thesis models never saw a real photo during training, and the synthetic-to-real gap is the main error source.
How much does CER drop if the best synthetic-trained adapter (Qwen2.5-VL-7B) is fine-tuned further on **N real lines**?

**Protocol.**
* 5 folds over the gold set, **grouped by inscription**: crops of one stone never appear in both train and test.
* For N ∈ {0, 10, 25, 50, all ≈ 90}: continue training the published adapter on N lines from the training folds,
  then predict the held-out fold. N = 0 is the adapter as published.
* Training subsets are nested (the 10 lines are part of the 25, and so on) and spread across as many inscriptions as possible.
* Every gold line is predicted exactly once per N. CER is pooled over all held-out lines, with 95 % CIs and
  paired tests against N = 0 from an inscription-level bootstrap.

**Robustness.** Every (N, fold) step runs in its **own Python process**, so GPU memory is fully released between steps.
Repeated load/train cycles in one kernel leaked memory and ended in CUDA OOM after six steps. A failed step is logged and
skipped instead of stopping the run; rerunning the notebook retries it.

**Output:** a learning curve (CER vs. number of real lines), per-fold predictions and `REPORT.md`.

**Runtime:** ≈ 21 steps × 3.5–6 min ≈ 1.5–2.5 h on 2× T4. Work is ordered N-outer / fold-inner, so a partial run already gives
complete points of the curve.

### How to run (fire and forget)
1. Kaggle → **New Notebook** → File → **Import Notebook** → this `.ipynb`.
2. Right panel: **Accelerator → GPU T4 x2 (or P100)**, **Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (Hugging Face token with *write* access). It is needed only to mirror
   results to the private dataset `AntoniusPerf/runic-ocr-results`; without it, results stay in the Output tab.
4. **Add Input** (optional, recommended): your dataset `zhopa228/val-dataset`. It has the one gold image missing from the
   repo copy, so evaluation covers 113 lines instead of 112.
5. **Save Version → Save & Run All (Commit)**, then close the tab. When the run finishes, open **Output**: everything is in
   `results/E2_fewshot_cv/`, with the summary in `REPORT.md`.

If the session stops early (time limit, GPU error), just run it again: finished steps are skipped. To resume **without** `HF_TOKEN`, add the previous version's Output as an Input
   (**Add Input → Your Work → this notebook**): finished steps are copied from there.

**Check the setup first (optional, 1 min, CPU):** set `DRY_RUN = True` and run all. A fake model replaces the GPU parts, so every step, table and chart is exercised. Set it back to `False` for the real run.

In [ ]:
# ── Configuration (the only cell you may want to edit) ──
REPO_URL = "https://github.com/kekys778/RUNIC-OCR.git"
REPO_REF = "main"

ADAPTER_REPO = "AntoniusPerf/runic-ocr-qwen-vl-lora"
MODEL_KEY = "qwen25vl-7b"  # best thesis model
K_FOLDS = 5
N_TRAIN = [
    10,
    25,
    50,
    -1,
]  # real lines per fold for fine-tuning; -1 = all lines of the training folds
EPOCHS = {10: 10, 25: 8, 50: 6, -1: 5}  # more passes for tiny sets; ≈ 25–110 optimizer steps each
LR = 5e-5  # lower than the thesis 1e-4: continued training of an already-trained adapter
GRAD_ACCUM = 4
SEED = 0
TIME_BUDGET_H = 11.0  # stop cleanly before Kaggle's 12 h limit; rerun to continue
STEP_RESERVE_H = 0.4  # do not start a new step if less than this is left
RESULTS_HF_REPO = "AntoniusPerf/runic-ocr-results"
DRY_RUN = False  # True: 1-minute CPU check of the whole pipeline with a fake model

In [ ]:
# ── Bootstrap: dependencies, code, sanity checks (≈2 min) ── no need to edit ──
import subprocess
import sys
import time
from pathlib import Path

T0 = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORK / "results"
REPO = Path("/tmp/RUNIC-OCR")  # outside /kaggle/working, so it is not saved as output

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "bitsandbytes>=0.46.1",
        "qwen-vl-utils",
        "rapidfuzz",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "tabulate"], check=True)
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)],
        check=True,
    )
sys.path.insert(0, str(REPO / "src"))

import torch  # noqa: E402

from runic_kaggle import (  # noqa: E402
    ResultSink,
    environment_report,
    find_adapter,
    get_secret,
    load_gold,
    run_isolated,
)

assert DRY_RUN or torch.cuda.is_available(), (
    "No GPU. Kaggle: Settings → Accelerator → GPU T4 x2 (or P100), then run again."
)
HF_TOKEN = get_secret("HF_TOKEN")
if not HF_TOKEN:
    print(
        "⚠ No HF_TOKEN secret: the public adapters still download, but results are only saved locally."
    )
ENV = environment_report()
print(ENV)

## 1. Gold set, folds, adapter

In [ ]:
import pandas as pd
from huggingface_hub import snapshot_download

import runic_ocr_experiments as H
import runic_reports as R
from runic_eval import grouped_kfold, nested_subsets

gold = load_gold(image_roots=["/kaggle/input"])
folds = grouped_kfold(gold["inscription"], k=K_FOLDS, seed=SEED)
fold_of = {int(i): f for f, (_, te) in enumerate(folds) for i in te}
ADAPTER = (
    "dry-run"
    if DRY_RUN
    else find_adapter(
        Path(snapshot_download(ADAPTER_REPO, token=HF_TOKEN)), H.EXPERIMENTS[MODEL_KEY].model_id
    )
)
STEP_DIR = WORK / "_steps"  # temporary spec/result files of the isolated steps


def step_spec(train_idx, test_idx):
    """Spec for one isolated step (see src/runic_step.py)."""
    paths = gold["path"].tolist()
    return {
        "model_key": MODEL_KEY,
        "adapter": str(ADAPTER),
        "train_items": [{"image": paths[i], "answer": gold["gt"].iloc[i]} for i in train_idx],
        "test_paths": [paths[i] for i in test_idx],
        "epochs": EPOCHS.get(len(train_idx), None) or EPOCHS.get(-1),
        "lr": LR,
        "grad_accum": GRAD_ACCUM,
        "seed": SEED,
        "trainer_dir": str(WORK / "_trainer_tmp"),
        "dry_run": DRY_RUN,
        "dry_gt": dict(zip(paths, gold["gt"])) if DRY_RUN else None,
    }


sink = ResultSink(
    "E2_fewshot_cv" + ("_dryrun" if DRY_RUN else ""),
    OUT_DIR,
    hf_repo="" if DRY_RUN else RESULTS_HF_REPO,
    hf_token=HF_TOKEN,
    seed_from=["/kaggle/input"],
)
sink.save_json(
    "setup.json",
    {
        **ENV,
        "repo_ref": REPO_REF,
        "model": MODEL_KEY,
        "adapter": str(ADAPTER),
        "k_folds": K_FOLDS,
        "n_train": N_TRAIN,
        "epochs": {str(k): v for k, v in EPOCHS.items()},
        "lr": LR,
        "grad_accum": GRAD_ACCUM,
        "dry_run": DRY_RUN,
        "folds": [
            {
                "fold": f,
                "test_lines": len(te),
                "test_inscriptions": sorted(set(gold["inscription"].iloc[te])),
            }
            for f, (_, te) in enumerate(folds)
        ],
    },
)
for f, (tr, te) in enumerate(folds):
    print(
        f"fold {f}: train {len(tr)} lines / test {len(te)} lines ({gold['inscription'].iloc[te].nunique()} inscriptions)"
    )
print("already done:", sorted(sink.done_steps()))

## 2. N = 0: the published adapter on every line (reference point)

In [ ]:
if not sink.is_done("N0"):
    res = run_isolated(step_spec([], range(len(gold))), STEP_DIR)
    df0 = R.predictions_frame(
        gold, res["recs"], fold=[fold_of[i] for i in range(len(gold))], n_train=0
    )
    sink.save_csv("pred_N0.csv", df0)
    sink.mark_done("N0", CER=round(R.score(df0, SEED)["CER"], 2), **res["info"])
print("N = 0 CER:", round(pd.read_csv(sink.path("pred_N0.csv")).pipe(R.score)["CER"], 2))

## 3. Fine-tune on N real lines, predict the held-out fold
Each step starts from the published adapter in a fresh process. A failed step is logged and skipped; rerun the notebook to retry it.

In [ ]:
stopped, failed = False, []
for n in N_TRAIN:
    for f, (tr, te) in enumerate(folds):
        step = f"N{n}_fold{f}"
        if sink.is_done(step):
            continue
        left = TIME_BUDGET_H - (time.time() - T0) / 3600
        if left < STEP_RESERVE_H:
            print(
                f"⏸ {left:.1f} h left of the budget: stopping cleanly. Run the notebook again to continue."
            )
            stopped = True
            break
        sub = nested_subsets(tr, N_TRAIN, groups=gold["inscription"].to_numpy(), seed=SEED + f)[n]
        spec = step_spec(sub, te)
        spec["epochs"] = EPOCHS[n]
        print(f"\n▶ {step}: {len(sub)} train lines → {len(te)} test lines")
        try:
            res = run_isolated(spec, STEP_DIR)
        except RuntimeError as e:
            print(f"  ✗ {step} failed, skipped (rerun the notebook to retry):\n{e}")
            failed.append(step)
            continue
        df = R.predictions_frame(gold.iloc[te], res["recs"], fold=f, n_train=len(sub))
        sink.save_csv(f"pred_{step}.csv", df)
        cer = R.score(df, SEED)["CER"]
        print(f"  fold CER {cer:.2f} %  | {res['info']}")
        sink.mark_done(step, CER=round(cer, 2), n_train=len(sub), **res["info"])
    if stopped:
        break
print("failed steps:", failed or "none")

## 4. Learning curve and report

In [ ]:
by_n = {0: pd.read_csv(sink.path("pred_N0.csv"))}
partial = []
for n in N_TRAIN:
    parts = [sink.path(f"pred_N{n}_fold{f}.csv") for f in range(K_FOLDS)]
    have = [pd.read_csv(p) for p in parts if p.exists()]
    if len(have) == K_FOLDS:
        by_n[n] = pd.concat(have, ignore_index=True)
    elif have:
        partial.append(f"N={n}: {len(have)}/{K_FOLDS} folds")

curve = R.learning_curve_table(by_n, seed=SEED)
sink.save_csv("learning_curve.csv", curve)
R.plot_learning_curve(
    curve,
    sink.path("learning_curve.png"),
    f"{MODEL_KEY}: CER on held-out inscriptions vs. real lines used for fine-tuning",
)

complete = not (partial or stopped or failed)
status = (
    "All steps finished."
    if complete
    else (
        "**Incomplete:** "
        + "; ".join(partial + [f"failed: {', '.join(failed)}"] * bool(failed))
        + ". Run the notebook again to finish."
    )
)
report = f"""# E2 · Few-shot fine-tuning on real photos ({MODEL_KEY}){" — DRY RUN (fake model)" if DRY_RUN else ""}

{K_FOLDS}-fold cross-validation grouped by inscription · {len(gold)} gold lines / {gold["inscription"].nunique()} inscriptions ·
lr {LR}, epochs {EPOCHS} · GPU {ENV.get("gpus")} · {(time.time() - T0) / 3600:.1f} h this session.
{status}

CER is pooled over all held-out lines. The CI and the Δ vs. N = 0 come from a paired bootstrap over inscriptions.

{R.df_to_md(curve)}

![learning curve](learning_curve.png)

Per-step logs: `run_log.jsonl` · predictions: `pred_N*_fold*.csv` · folds: `setup.json`.
"""
sink.save_text("REPORT.md", report)
sink.mark_done("summary" if complete else f"partial_summary_{int(time.time())}")
print(report)